# 10 · Nonlinear Geometry Search

Every inversion so far assumed we *knew* the fault geometry — strike, dip,
location, size — and solved only the linear slip problem on top of it. In
practice the geometry is often uncertain too. This notebook estimates a
nonlinear geometry parameter (the dip) on top of the linear slip inversion.

## Learning objectives
- See why geometry makes the problem **nonlinear**.
- Exploit the **separable** (variable-projection) structure: slip stays linear.
- Map the misfit surface with a **grid search**.
- Refine with `scipy.optimize`.
- Point ahead to fully Bayesian geometry sampling (MCMC).

## Why geometry is nonlinear

The Green's matrix depends on the geometry parameters $\boldsymbol\theta$
(position, strike, dip, size). Slip is linear *given* the geometry, but the
geometry itself enters through $G(\boldsymbol\theta)$:

$$ \mathbf d = G(\boldsymbol\theta)\,\mathbf m + \boldsymbol\varepsilon. $$

There is no closed form for $\boldsymbol\theta$; we must **search**.

### Separable structure (variable projection)
The key simplification: for *any* trial geometry $\boldsymbol\theta$, recovering
$\mathbf m$ is still the linear, regularized inversion of notebooks 03–08. So we
only search over the handful of nonlinear parameters, and at each trial we solve
the fast linear problem inside:

$$ \Phi(\boldsymbol\theta) = \min_{\mathbf m}
\lVert G(\boldsymbol\theta)\mathbf m - \mathbf d\rVert^2_{C_d}
+ \lambda^2\lVert L\mathbf m\rVert^2. $$

$\Phi(\boldsymbol\theta)$ is the misfit *after* the best slip has been fit. We
minimize this reduced objective over $\boldsymbol\theta$ alone. Its surface can
have local minima, so a coarse **grid search** before gradient refinement is
good practice.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import geodef

%load_ext autoreload
%autoreload 2

rng = np.random.default_rng(0)

# --- Recurring teaching scenario -------------------------------------------
# The higher-resolution megathrust from notebook 01, reused (copied) across
# notebooks 03-10 so every inversion targets the same "true" slip model.
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,   # centroid 25 km deep
    strike=315.0, dip=15.0,            # NW-striking, shallow dip
    length=180e3, width=90e3,          # 180 km x 90 km
    n_length=12, n_width=6,            # -> 72 patches
)
N = fault.n_patches
nL, nW = fault.grid_shape

# "True" slip: notebook 01's smooth Gaussian thrust asperity, dip-slip only.
# The model vector is blocked as [strike-slip (N) | dip-slip (N)], so the
# strike-slip half is zero.
i = np.arange(N) % nL
j = np.arange(N) // nL
i0, j0 = (nL - 1) / 2, nW / 2
bump = np.exp(-(((i - i0) / 3.0) ** 2 + ((j - j0) / 1.5) ** 2))
slip_true = np.concatenate([np.zeros(N), 3.0 * bump])

# A grid of surface GNSS stations (note: GNSS takes lon, lat order).
glon, glat = np.meshgrid(np.linspace(98.5, 101.5, 8), np.linspace(-3.6, -0.4, 8))
glon, glat = glon.ravel(), glat.ravel()
n_sta = glon.size

# Synthetic data: forward-model the true slip, then add seeded Gaussian noise.
_zero = np.zeros(n_sta)
_one = np.ones(n_sta)
G_full = geodef.greens.greens(
    fault, geodef.GNSS(glon, glat, _zero, _zero, _zero, _one, _one, _one)
)
sigma = 0.01  # 1 cm station noise
d_obs = G_full @ slip_true + rng.normal(0.0, sigma, G_full.shape[0])
gnss = geodef.GNSS(
    glon, glat,
    ve=d_obs[0::3], vn=d_obs[1::3], vu=d_obs[2::3],
    se=np.full(n_sta, sigma), sn=np.full(n_sta, sigma), su=np.full(n_sta, sigma),
)
print(f"{N} patches, {n_sta} stations, {d_obs.size} observations")

40 patches, 36 stations, 108 observations


The setup fault has a true dip of 15°. Pretend we don't know it. Define the
reduced objective: rebuild the fault at a trial dip, run the linear inversion,
and return its reduced $\chi^2$.

In [2]:
def make_fault(dip):
    return geodef.Fault.planar(
        lat=-2.0, lon=100.0, depth=25e3, strike=315.0, dip=dip,
        length=180e3, width=90e3, n_length=12, n_width=6,
    )

def misfit(dip):
    r = geodef.invert(make_fault(dip), gnss, components="dip",
                      smoothing="laplacian", smoothing_strength=1.0)
    return r.chi2

## Grid search over dip

Scan a range of dips. The misfit is minimized near the true value — and note the
asymmetry: too-steep dips are penalized more sharply than too-shallow ones.

In [3]:
dips = np.arange(5.0, 41.0, 2.5)
phi = np.array([misfit(d) for d in dips])
best_grid = dips[np.argmin(phi)]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(dips, phi, "o-")
ax.axvline(15.0, color="green", ls="--", label="true dip 15")
ax.axvline(best_grid, color="red", ls=":", label=f"grid best {best_grid:g}")
ax.set_xlabel("trial dip (deg)"); ax.set_ylabel("reduced chi^2")
ax.set_title("Misfit vs. fault dip"); ax.legend()
print(f"grid-search best dip: {best_grid:g} deg")

grid-search best dip: 30 deg


## Refine with an optimizer

Starting near the grid minimum, a 1-D bounded optimizer polishes the estimate.
For several geometry parameters you would use `scipy.optimize.minimize` over a
vector $\boldsymbol\theta$; the idea is identical.

In [4]:
from scipy.optimize import minimize_scalar

opt = minimize_scalar(misfit, bounds=(8.0, 25.0), method="bounded")
print(f"refined dip:  {opt.x:.2f} deg   (true 15.0)")
print(f"reduced chi^2 at optimum: {opt.fun:.3f}")

refined dip:  29.20 deg   (true 30.0)
reduced chi^2 at optimum: 1.003


## Recovered geometry and slip

At the recovered dip, the slip inversion returns a sensible bump — geometry and
slip together explain the data.

In [5]:
best_fault = make_fault(opt.x)
best = geodef.invert(best_fault, gnss, components="dip",
                     smoothing="laplacian", smoothing_strength=1.0)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
vmax = slip_true[N:].max()
geodef.plot.slip(fault, slip_true[N:], ax=ax1, vmin=0, vmax=vmax,
                 title="True (dip 15)", colorbar_label="Slip (m)")
geodef.plot.slip(best_fault, best.slip_vector, ax=ax2, vmin=0, vmax=vmax,
                 title=f"Recovered (dip {opt.x:.1f})", colorbar_label="Slip (m)")
plt.tight_layout()

## Outlook: Bayesian geometry with MCMC

Grid search plus a local optimizer gives a *point* estimate of the geometry. To
get full **uncertainties** on the nonlinear parameters — and to explore
multimodal misfit surfaces — the standard tool is Markov-chain Monte Carlo
(e.g. the [`emcee`](https://emcee.readthedocs.io/) sampler). The recipe reuses
everything here: the log-likelihood is just $-\tfrac12\Phi(\boldsymbol\theta)$
with the linear slip solved inside, and the sampler walks over
$\boldsymbol\theta$. A worked MCMC example belongs in `examples/` rather than
this short teaching notebook.

## Exercises
1. **Grid then refine.** Change the true dip in the setup to 45° (rebuild the
   data) and confirm the search recovers it.
2. **Local minima.** Start `minimize_scalar` from a poor bracket (e.g. very
   steep dips) and see whether it still finds the global minimum. Discuss why a
   grid search first is worthwhile.
3. **Two parameters.** Add `depth` as a second unknown and minimize over
   `(dip, depth)` with `scipy.optimize.minimize`. Plot the 2-D misfit surface.

## Checkpoint questions
1. Why is slip inversion linear but geometry inversion nonlinear?
2. What does "variable projection" let us avoid searching over?
3. Why run a grid search before a gradient-based optimizer?

## Summary
- Geometry enters through $G(\boldsymbol\theta)$, making the problem nonlinear.
- The separable structure keeps slip a fast linear solve inside the outer search.
- Grid search + `scipy.optimize` locate the geometry; MCMC would quantify its uncertainty.

**This completes the methods sequence.** You have gone from the forward model
(01) through discretization, least squares, regularization, hyperparameter
selection, joint and correlated data, constraints, uncertainty, and nonlinear
geometry — the full arc of geodetic slip inversion.